# Distributed Training with DDP

In this lab, we spread a model's training over several processes with `DistributedDataParallel` (DDP), PyTorch's standard mechanism for **data parallelism**.

Colab only provides one GPU: we will therefore launch 2 processes **on CPU**, communicating through the `gloo` backend. On a machine with 4 or 8 GPUs, the same code only changes in two places: the backend (`nccl`), and placing the model on each process's GPU (its number on the machine, `LOCAL_RANK`, given to `torch.cuda.set_device`, to `.to()` and to the `device_ids` of `DDP`).

Outline:

1. write a DDP training script and launch it with `torchrun`;
2. check that the processes stay synchronized;
3. save a checkpoint correctly;
4. do the same in a few lines with PyTorch Lightning;
5. optionally, train with mixed precision.

In [ ]:
!pip install -q lightning

## The starting point

Here is a classic, single-process training script for a synthetic classification problem. The `%%writefile` magic writes the cell's content to a file instead of running it.

In [ ]:
%%writefile common.py
import torch
from torch import nn
from torch.utils.data import TensorDataset


def make_dataset(n: int = 8192) -> TensorDataset:
  # Same seed in every process: they all see the same dataset
  generator = torch.Generator().manual_seed(0)
  x = torch.randn(n, 20, generator=generator)
  true_weights = torch.randn(20, 3, generator=generator)
  y = (x @ true_weights).argmax(dim=1)
  return TensorDataset(x, y)


def make_model() -> nn.Module:
  torch.manual_seed(0)
  return nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 3))

In [ ]:
%%writefile train_single.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model

dataset = make_dataset()
model = make_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

for epoch in range(3):
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch}: {len(loader)} batches, last loss {loss.item():.3f}")

In [ ]:
!python train_single.py

## A DDP script

With DDP, `torchrun` launches several **processes** running the same script. Each process:

- joins the process group (`dist.init_process_group`); `torchrun` gives it its number (*rank*) and the total number of processes through environment variables;
- wraps its model in `DistributedDataParallel`, which averages gradients across processes during `backward()`;
- only reads its share of the dataset, thanks to a `DistributedSampler`.

*Complete the `train_ddp.py` script below (the `...`):*

- *initialize the process group with the `"gloo"` backend and get the rank and number of processes (`dist.get_rank()`, `dist.get_world_size()`);*
- *wrap the model in `DDP`;*
- *create a `DistributedSampler` and pass it to the `DataLoader` (instead of `shuffle=True`);*
- *call `sampler.set_epoch(epoch)` at the start of each epoch;*
- *only print messages from the rank 0 process.*

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

...  # Process group initialization, rank and number of processes

dataset = make_dataset()
model = ...  # Model wrapped in DDP
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = ...
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  ...
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  ...  # Print from rank 0 only

dist.destroy_process_group()

### Additional hints

- `dist.init_process_group("gloo")` is called before any other function of `dist`: `dist.get_rank()` and `dist.get_world_size()` depend on it.
- On CPU, `DDP` only takes the model: `DDP(make_model())`, without `device_ids`.
- The `DistributedSampler` shuffles the data itself (`shuffle=True` by default): the `DataLoader` refuses `shuffle=True` together with a `sampler`.
- `set_epoch(epoch)` is called before the loop over the batches, so that each epoch gets its own shuffle.

```python
dist.init_process_group(...)
rank = ...
world_size = ...

dataset = make_dataset()
model = DDP(...)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(...)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(...)
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  if ...:
    print(f"Epoch {epoch}: last loss {loss.item():.3f}")
```

### Solution

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

# On GPU, first this process's GPU, its number on the machine (LOCAL_RANK):
# local_rank = dist.get_node_local_rank(); torch.cuda.set_device(local_rank)
dist.init_process_group("gloo")  # "nccl" on GPU
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())  # On GPU: .to(local_rank), device_ids=[local_rank]
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  if rank == 0:
    print(
      f"Epoch {epoch}: {len(loader)} batches per process, "
      f"{world_size} processes, last loss {loss.item():.3f}"
    )

dist.destroy_process_group()

### Launching the script

Launch the script on 2 processes:

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

## Parallel training analysis

1. *How many batches does each process handle per epoch, compared with the single-process script? Why?*
2. *Each process uses batches of 64 examples. How many examples contribute to each weight update? What does it imply for the choice of learning rate?*
3. *What is `sampler.set_epoch(epoch)` for?*

*Your answers here.*

### Solution

1. Each process handles 64 batches per epoch instead of 128: the `DistributedSampler` gives each process a different half of the dataset. With 2 GPUs, an epoch therefore takes about half the time. Here, on CPU, the DDP script is slower than the single-process one instead: launching the processes, exchanging the gradients through `gloo` at every batch and a single thread per process (`torchrun` sets `OMP_NUM_THREADS=1`) cost more than sharing the data saves on such a small model.
2. Gradients are averaged across the 2 processes: each update uses 2 × 64 = 128 examples. The effective batch grows with the number of processes; the learning rate is often increased accordingly (with a *warmup*), otherwise training progresses less per epoch.
3. The `DistributedSampler` shuffles the data with a seed that depends on the epoch: without `set_epoch`, every epoch would see the examples in the same order.

## Do the processes stay synchronized?

Each process has its own copy of the model. DDP guarantees they stay identical: weights are broadcast from rank 0 when the `DDP` is created, then all processes apply the same update, computed from the averaged gradients.

*Let's check it. The `train_ddp_check.py` script below repeats the training; complete its end (the `...`): compute in each process the sum of all the model's weights, gather these sums with `dist.all_gather`, then print them from rank 0.*

**Useful documentation**

- [`torch.distributed.all_gather`](https://docs.pytorch.org/docs/stable/distributed.html#torch.distributed.all_gather)

In [ ]:
%%writefile train_ddp_check.py
import sys

import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

# Launched with --no-ddp, the script trains the model without wrapping it in DDP
with_ddp = "--no-ddp" not in sys.argv

dist.init_process_group("gloo")
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model()) if with_ddp else make_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    cross_entropy(model(x), y).backward()
    optimizer.step()

checksum = ...  # Sum of all the model's weights, in this process
checksums = ...  # The list dist.all_gather fills, one slot per process
...  # Gathering the sums of all processes
if rank == 0:
  print("Sum of the weights in each process:")
  print([f"{s.item():.6f}" for s in checksums])

dist.destroy_process_group()

### Additional hints

- `p.detach().sum()`: the sum of the elements of a tensor `p` of `model.parameters()`, outside the computation graph; `torch.stack(...)` puts these sums in one tensor, whose `.sum()` gives the sum of all the weights, a zero-dimensional tensor.
- `dist.all_gather(tensor_list, tensor)` fills `tensor_list`, which already holds one tensor of the same shape per process: `[torch.zeros(()) for _ in range(world_size)]`.
- All processes call `dist.all_gather`, not only rank 0: it is a collective operation, which waits for each of them.

```python
checksum = torch.stack([... for p in model.parameters()]).sum()
checksums = [torch.zeros(()) for _ in range(...)]
dist.all_gather(..., ...)
```

### Solution

In [ ]:
%%writefile train_ddp_check.py
import sys

import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

# Launched with --no-ddp, the script trains the model without wrapping it in DDP
with_ddp = "--no-ddp" not in sys.argv

dist.init_process_group("gloo")
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model()) if with_ddp else make_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    cross_entropy(model(x), y).backward()
    optimizer.step()

checksum = torch.stack([p.detach().sum() for p in model.parameters()]).sum()
checksums = [torch.zeros(()) for _ in range(world_size)]
dist.all_gather(checksums, checksum)
if rank == 0:
  print("Sum of the weights in each process:")
  print([f"{s.item():.6f}" for s in checksums])

dist.destroy_process_group()

### With and without DDP

Launch the script on 2 processes, then launch it again with `--no-ddp`: the model is no longer wrapped in `DDP`, and each process trains it alone.

*Are the sums equal in both cases? Why?*

In [ ]:
!torchrun --nproc_per_node=2 train_ddp_check.py

In [ ]:
!torchrun --nproc_per_node=2 train_ddp_check.py --no-ddp

*Your answers here.*

#### Solution

With DDP, both sums are identical: the copies start from the same weights, broadcast from rank 0, and all receive the same update, computed from the averaged gradients. Without DDP, they differ: both processes do start from the same weights (`make_model` sets the seed), but each one only sees its half of the data and applies its own gradients, without averaging them with the other's: the copies diverge from the first update on.

## Saving a checkpoint

In distributed training, all processes have the same weights: only one must write the checkpoint, otherwise they all write the same file at the same time.

*Modify `train_ddp.py` to save the model at the end of training, from rank 0 only, to `model.pt`. Then check, in a notebook cell, that you can reload these weights into a **non-distributed** model created with `make_model()`.*

*Hint: the original model is available as `model.module`. Why save `model.module.state_dict()` rather than `model.state_dict()`?*

In [ ]:
# Your code here

### Additional hints

- `model.module`: the original model, without the `DDP` wrapper.
- Saving comes after the training loop, inside an `if rank == 0:`, before `dist.destroy_process_group()`.
- In the notebook, `load_state_dict` takes the dictionary `torch.load` reads back.

```python
if rank == 0:  # In train_ddp.py, after the training loop
  torch.save(..., "model.pt")

reloaded = make_model()  # In a notebook cell
reloaded.load_state_dict(...)
```

### Solution

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

dist.init_process_group("gloo")
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  if rank == 0:
    print(f"Epoch {epoch}: last loss {loss.item():.3f}")

if rank == 0:
  torch.save(model.module.state_dict(), "model.pt")
  print("Checkpoint saved")

dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

In [ ]:
import torch

from common import make_model

reloaded = make_model()
reloaded.load_state_dict(torch.load("model.pt"))
print("Weights reloaded into a regular model")

`model.state_dict()` would prefix every key with `module.` (the name of the attribute where DDP stores the model): these weights couldn't be reloaded into a regular model without renaming the keys.

## With PyTorch Lightning

Lightning handles everything we wrote by hand: process group, `DDP`, `DistributedSampler`, `set_epoch`, printing and saving from rank 0. You just pick a strategy in the `Trainer`.

*Complete the `Trainer` of `train_lightning.py` to train on `world_size` CPU processes, the number given to the script (`python train_lightning.py 1` for a single one, 2 without an argument), with the `"ddp"` strategy.*

In [ ]:
%%writefile train_lightning.py
import sys

import lightning
import torch
from lightning.pytorch.callbacks import TQDMProgressBar
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model


class Classifier(lightning.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.model = make_model()

  def training_step(self, batch, batch_index):
    x, y = batch
    loss = cross_entropy(self.model(x), y)
    self.log("train_loss", loss, prog_bar=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  # The number given to the script, 2 without an argument
  world_size = int(sys.argv[1]) if len(sys.argv) > 1 else 2
  loader = DataLoader(make_dataset(), batch_size=64, shuffle=True)
  trainer = lightning.Trainer(
    max_epochs=3,
    ...,  # Your code here
    # tqdm's progress bar, rather than rich's, which Lightning picks when rich
    # is installed (Colab): it froze some trainings
    callbacks=[TQDMProgressBar()],
    # No checkpoint nor log: this lab reads none back
    enable_checkpointing=False,
    logger=False,
  )
  trainer.fit(Classifier(), loader)
  if trainer.is_global_zero:
    print(f"{trainer.num_training_batches} steps per epoch and per process")

### Additional hints

- `accelerator="cpu"`: processes on CPU, like our DDP script.
- `devices=world_size`: the number of processes Lightning launches, 2 when the script is launched without an argument.
- `strategy="ddp"`: one `DistributedDataParallel` per process, with the `gloo` backend on CPU.

```python
trainer = lightning.Trainer(
  max_epochs=3,
  accelerator=...,
  devices=...,
  strategy=...,
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
  logger=False,
)
```

### Solution

In [ ]:
%%writefile train_lightning.py
import sys

import lightning
import torch
from lightning.pytorch.callbacks import TQDMProgressBar
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model


class Classifier(lightning.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.model = make_model()

  def training_step(self, batch, batch_index):
    x, y = batch
    loss = cross_entropy(self.model(x), y)
    self.log("train_loss", loss, prog_bar=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  # The number given to the script, 2 without an argument
  world_size = int(sys.argv[1]) if len(sys.argv) > 1 else 2
  loader = DataLoader(make_dataset(), batch_size=64, shuffle=True)
  trainer = lightning.Trainer(
    max_epochs=3,
    accelerator="cpu",
    devices=world_size,
    strategy="ddp",
    # tqdm's progress bar, rather than rich's, which Lightning picks when rich
    # is installed (Colab): it froze some trainings
    callbacks=[TQDMProgressBar()],
    # No checkpoint nor log: this lab reads none back
    enable_checkpointing=False,
    logger=False,
  )
  trainer.fit(Classifier(), loader)
  if trainer.is_global_zero:
    print(f"{trainer.num_training_batches} steps per epoch and per process")

### On one then two processes

Launch the script on 1 process, then without an argument, on 2. At the end, it prints the number of steps (of batches) each process takes per epoch.

In [ ]:
!python train_lightning.py 1

In [ ]:
!python train_lightning.py

1. *How many steps per epoch does each process take, on 1 then on 2 processes? Why, given that the `DataLoader` is created with `shuffle=True` and no `DistributedSampler`?*
2. *On a machine with 4 GPUs, what would need to change in the `Trainer`?*

*Your answers here.*

#### Solution

1. 128 steps per epoch on 1 process (8,192 examples in batches of 64), 64 on 2: with the `"ddp"` strategy, Lightning replaces the `DataLoader`'s `shuffle=True` with a `DistributedSampler`, whose `set_epoch` it calls itself, and each process only sees half of the data, as in our DDP script.
2. `accelerator="gpu", devices=4` is enough (plus `precision="bf16-mixed"` for mixed precision): Lightning picks the `nccl` backend and puts each process on its GPU (`LOCAL_RANK`, `torch.cuda.set_device`).

## Mixed precision (optional)

This last, optional part goes back to the single-process script of the beginning, `train_single.py`. Mixed precision (the course's "Mixed precision" slide) runs the forward pass in 16 bits, but keeps the weights in 32 bits; it combines with DDP without changing anything in it (in Lightning, `precision="bf16-mixed"`). On GPU, it speeds training up markedly; on CPU, only on processors with bfloat16 instructions: here, we look at the tensors' types, not at the time.

*Modify `train_single.py` below to compute the model's output and the loss in bfloat16 with `torch.autocast`, on CPU. At the end of training, print the type of the model's output, of the loss, of the first layer's weights and of their gradients.*

In [ ]:
%%writefile train_single.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model

dataset = make_dataset()
model = make_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

for epoch in range(3):
  for x, y in loader:
    optimizer.zero_grad()
    output = model(x)  # The output and the loss, in bfloat16
    loss = cross_entropy(output, y)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch}: {len(loader)} batches, last loss {loss.item():.3f}")

...  # Types of the output, the loss, the weights and their gradients

### Additional hints

- `with torch.autocast(device_type="cpu", dtype=torch.bfloat16):` encloses the forward pass and the loss; `loss.backward()` and `optimizer.step()` stay outside the block, as on the "Mixed precision" slide.
- `model` is an `nn.Sequential`: `model[0]` is its first layer, `model[0].weight` its weights, and `model[0].weight.grad` their gradients, which `backward()` fills in.
- A tensor's type: its `.dtype` attribute.

```python
for epoch in range(3):
  for x, y in loader:
    optimizer.zero_grad()
    with torch.autocast(device_type=..., dtype=...):
      output = model(x)
      loss = ...
    loss.backward()
    optimizer.step()
  print(...)  # Unchanged

print(f"Model output: {output.dtype}")
print(f"Loss: {...}")
print(f"First layer weights: {...}")
print(f"Gradients of these weights: {...}")
```

### Solution

In [ ]:
%%writefile train_single.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model

dataset = make_dataset()
model = make_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

for epoch in range(3):
  for x, y in loader:
    optimizer.zero_grad()
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
      output = model(x)
      loss = cross_entropy(output, y)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch}: {len(loader)} batches, last loss {loss.item():.3f}")

print(f"Model output: {output.dtype}")
print(f"Loss: {loss.dtype}")
print(f"First layer weights: {model[0].weight.dtype}")
print(f"Gradients of these weights: {model[0].weight.grad.dtype}")

### Types in mixed precision

Launch the script, which now trains with mixed precision:

In [ ]:
!python train_single.py

1. *What type does the model's output have, and what type does the loss have? Why aren't they the same?*
2. *Have the weights and their gradients switched to bfloat16? Why is that preferable?*
3. *With `dtype=torch.float16`, a `torch.amp.GradScaler` would be needed. Why isn't one needed with bfloat16?*

*Your answers here.*

#### Solution

1. The output is in `bfloat16`: inside the block, autocast computes the matrix products of the `nn.Linear` layers in bfloat16, and `ReLU` keeps its input's type. The loss is in `float32`: `cross_entropy` is one of the operations autocast computes in float32, because its exponentials, logarithms and sums lose too much precision in 16 bits.
2. No: autocast doesn't convert the parameters, it computes each matrix product with a bfloat16 copy of the weights. The weights stay in float32, and so do their gradients, stored in each parameter's `.grad`. That is on purpose: an update (learning rate × gradient) is often very small compared with the weight, and bfloat16 only has about 3 significant digits. In bfloat16, `1 + 0.001` equals `1`: the update would be lost.
3. bfloat16 keeps float32's 8 exponent bits, hence the same range of values: it loses significant digits, but a small gradient doesn't become zero. float16 only has 5 exponent bits: very small gradients round to zero in it. `torch.amp.GradScaler` protects them, by multiplying the loss before `backward()`, then dividing the gradients before the update.

## To go further

- If the model doesn't fit in one GPU's memory, DDP is no longer enough: the `"fsdp"` strategy (*Fully Sharded Data Parallel*) also shards weights, gradients and optimizer states across GPUs.
- On several machines, the same command is launched on each one, with the number of machines and the address of one of them, where the processes meet: `torchrun --nnodes=2 --nproc_per_node=4 --rdzv_id=ddp_lab --rdzv_backend=c10d --rdzv_endpoint=machine0:29500 train_ddp.py`.
- See [PyTorch's DDP tutorial](https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html) and the [`torchrun` documentation](https://docs.pytorch.org/docs/stable/elastic/run.html).